# Analysis — Generate All Figures (ImageNet-100)
**Run after all 12 training runs are complete.**
Takes ~3h for analysis + seconds for plotting.


In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================
# EXTRACT IMAGENET-100 + RESIZE TO 256px
# ============================================

import os, time, tarfile, shutil, subprocess
from PIL import Image

DRIVE_BASE = '/content/drive/My Drive/pe_experiment'
RESULTS_DIR = os.path.join(DRIVE_BASE, 'results')
SCRIPT_DIR = DRIVE_BASE
DATA_DIR = '/content/imagenet100_resized'
IN100_RAW = '/content/imagenet100_raw'

train_path = os.path.join(DATA_DIR, 'train')
val_path = os.path.join(DATA_DIR, 'val')

if (os.path.exists(train_path) and len(os.listdir(train_path)) >= 100
    and os.path.exists(val_path) and len(os.listdir(val_path)) >= 100):
    print(f'✅ ImageNet-100 already on SSD')
else:
    imagenet_dir = os.path.join(DRIVE_BASE, 'imagenet')
    TRAIN_TAR = None
    VAL_TAR = None
    for name in ['ILSVRC2012_img_train.tar', 'ILSVRC2012_img_train']:
        c = os.path.join(imagenet_dir, name)
        if os.path.exists(c): TRAIN_TAR = c; break
    for name in ['ILSVRC2012_img_val.tar', 'ILSVRC2012_img_val']:
        c = os.path.join(imagenet_dir, name)
        if os.path.exists(c): VAL_TAR = c; break
    print(f'Train tar: {TRAIN_TAR}')
    print(f'Val tar: {VAL_TAR}')

    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/HobbitLong/CMC/master/imagenet100.txt',
        '-O', '/content/imagenet100.txt'], check=True)
    with open('/content/imagenet100.txt') as f:
        class_set = set(line.strip() for line in f if line.strip())
    print(f'Target classes: {len(class_set)}')

    t0 = time.time()

    print('Step 1/3: Extracting train...')
    raw_train = os.path.join(IN100_RAW, 'train')
    os.makedirs(raw_train, exist_ok=True)
    tf = tarfile.open(TRAIN_TAR, 'r|')
    found = 0
    for member in tf:
        if not member.name.endswith('.tar'): continue
        class_name = os.path.basename(member.name).replace('.tar', '')
        if class_name in class_set:
            class_dir = os.path.join(raw_train, class_name)
            os.makedirs(class_dir, exist_ok=True)
            fileobj = tf.extractfile(member)
            with tarfile.open(fileobj=fileobj, mode='r|') as ctf:
                ctf.extractall(class_dir)
            found += 1
            if found % 10 == 0: print(f'  {found}/100 classes ({time.time()-t0:.0f}s)')
            if found >= len(class_set): break
        else:
            tf.extractfile(member).read()
    tf.close()
    print(f'  Train: {found} classes')

    print('Step 2/3: Extracting val...')
    val_tmp = '/content/val_tmp'
    os.makedirs(val_tmp, exist_ok=True)
    with tarfile.open(VAL_TAR, 'r|') as vtf:
        vtf.extractall(val_tmp)
    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/tensorflow/models/master/research/slim/datasets/imagenet_2012_validation_synset_labels.txt',
        '-O', '/content/val_synsets.txt'], check=True)
    with open('/content/val_synsets.txt') as f:
        val_synsets = [line.strip() for line in f]
    raw_val = os.path.join(IN100_RAW, 'val')
    os.makedirs(raw_val, exist_ok=True)
    val_images = sorted([f for f in os.listdir(val_tmp) if f.endswith('.JPEG')])
    for img_name, synset in zip(val_images, val_synsets):
        if synset in class_set:
            class_dir = os.path.join(raw_val, synset)
            os.makedirs(class_dir, exist_ok=True)
            shutil.move(os.path.join(val_tmp, img_name), os.path.join(class_dir, img_name))
    shutil.rmtree(val_tmp)

    print('Step 3/3: Resizing to 256px...')
    for split in ['train', 'val']:
        src_split = os.path.join(IN100_RAW, split)
        dst_split = os.path.join(DATA_DIR, split)
        for cls in os.listdir(src_split):
            cls_src = os.path.join(src_split, cls)
            cls_dst = os.path.join(dst_split, cls)
            os.makedirs(cls_dst, exist_ok=True)
            for img_name in os.listdir(cls_src):
                dst_p = os.path.join(cls_dst, img_name)
                if not os.path.exists(dst_p):
                    img = Image.open(os.path.join(cls_src, img_name)).convert('RGB')
                    img.thumbnail((256, 256), Image.LANCZOS)
                    img.save(dst_p, 'JPEG', quality=95)
        print(f'  {split} done')
    shutil.rmtree(IN100_RAW)
    print(f'✅ Done in {(time.time()-t0)/60:.1f} min')


In [ ]:
!pip install -q timm tqdm scikit-learn scipy matplotlib


In [ ]:
# Copy and verify script
import shutil
shutil.copy2(os.path.join(SCRIPT_DIR, 'full_scale_experiment.py'), '/content/')
!grep '_orig_mod\|max_components\|min_epochs\|ImageNet-100' /content/full_scale_experiment.py | head -6
print('✅ Latest script confirmed')


In [ ]:
# Verify all 12 models
import os, json
for pe in ['learned', 'sinusoidal', 'rope', 'alibi']:
    for s in [42, 123, 456]:
        path = os.path.join(RESULTS_DIR, f'{pe}_seed{s}')
        has_best = os.path.exists(os.path.join(path, 'best_model.pth'))
        has_hist = os.path.exists(os.path.join(path, 'training_history.json'))
        acc = ''
        if has_hist:
            with open(os.path.join(path, 'training_history.json')) as f:
                h = json.load(f)
            acc = f'best={max(h["val_acc"]):.2f}%'
        status = '✅' if has_best and has_hist else '❌'
        print(f'{status} {pe:12s} seed={s:3d}  {acc}')


## 🚀 Run Analysis (~3h)


In [ ]:
# Run full analysis + figure generation
!python /content/full_scale_experiment.py \
    --data_dir /content/imagenet100_resized \
    --output_dir "/content/drive/My Drive/pe_experiment/results" \
    --mode analyze \
    --num_classes 100


In [ ]:
# Display generated figures
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

fig_dir = os.path.join(RESULTS_DIR, 'figures')
if os.path.exists(fig_dir) and os.listdir(fig_dir):
    for fig_name in sorted(os.listdir(fig_dir)):
        if fig_name.endswith('.png'):
            img = mpimg.imread(os.path.join(fig_dir, fig_name))
            plt.figure(figsize=(16, 8))
            plt.imshow(img); plt.axis('off'); plt.title(fig_name)
            plt.show()
else:
    print('No figures found — check for errors above')
